# 读取 data/ 下 9 个 .db
每个 db 一张表 `t`,列名取自 `storage.SCHEMAS`(第一列是 `时间`,其余是指标)。

In [1]:
import sqlite3
from pathlib import Path
import pandas as pd

DATA_DIR = next((p for p in [Path("data"), Path("../data")]
                 if p.is_dir() and any(p.glob("*.db"))), Path("data"))  # notebook 在 archive/,data 在项目根,两种 cwd 都兜住

In [2]:
def load_all(data_dir: Path = DATA_DIR) -> dict[str, pd.DataFrame]:
    """把 data/ 下所有 .db 读成 {源名: DataFrame},时间列转成 datetime。"""
    out: dict[str, pd.DataFrame] = {}
    for db_path in sorted(data_dir.glob("*.db")):
        name = db_path.stem
        with sqlite3.connect(db_path) as conn:
            df = pd.read_sql_query("SELECT * FROM t", conn)
        df["时间"] = pd.to_datetime(df["时间"])
        out[name] = df
    return out

dfs = load_all()
print(f"已加载 {len(dfs)} 个源: {list(dfs)}")

已加载 9 个源: ['12378', '12378明细', '会话记录', '在线', '工单明细', '常规', '热线', '热线明细', '贷后']


In [3]:
# 辅助:按时间范围过滤某源
def slice_source(name: str, start: str, end: str) -> pd.DataFrame:
    df = dfs[name]
    return df[(df["时间"] >= start) & (df["时间"] <= end)]

In [4]:
# 每个源的行数 + 列名预览
for name, df in dfs.items():
    print(f"{name:8s}  rows={len(df):4d}  cols={list(df.columns)}")

12378     rows=3445  cols=['时间', '转人工量', '接通量', '排队量', '累计呼入量']
12378明细   rows=3445  cols=['时间', '签入', '通话', '空闲', '离席', '话后', '振铃', '置忙']
会话记录      rows= 828  cols=['时间', '转接一组', '转接二组', '贷后转接组']
在线        rows=3498  cols=['时间', '转人工量', '转人工失败', '排队', '咨询', '在线', '小休', '示忙', '话后', '就餐', '培训', '回访']
工单明细      rows= 828  cols=['时间', '二线客诉处理组', '常规工单处理组', '回访组一组', '贷后回访组', '12378回访组']
常规        rows=3498  cols=['时间', '签入', '通话', '空闲', '离席', '话后', '振铃', '置忙']
热线        rows=3505  cols=['时间', '转人工量', '接通量', '排队量', '累计呼入量', '外呼量', '外呼接通量']
热线明细      rows=3499  cols=['时间', '签入', '通话', '空闲', '离席', '话后', '振铃', '置忙']
贷后        rows=3486  cols=['时间', '签入', '通话', '空闲', '离席', '话后', '振铃', '置忙']


In [5]:
start_time = "2026-07-01 08:30"
end_time = "2026-07-29 21:00"

In [6]:
slice_source("热线", start_time, end_time).sort_values("时间", ascending=False).to_excel(r"D:\ucredit\liyuting\Desktop\热线.xlsx", index=False)
slice_source("在线", start_time, end_time).sort_values("时间", ascending=False).to_excel(r"D:\ucredit\liyuting\Desktop\在线.xlsx", index=False)

In [ ]:
slice_source("热线", start_time, end_time).sort_values("时间", ascending=False).head()

In [ ]:
slice_source("热线明细", start_time, end_time).sort_values("时间", ascending=False).head()

In [ ]:
slice_source("在线", start_time, end_time).sort_values("时间", ascending=False).head()

In [ ]:
slice_source("12378", start_time, end_time).sort_values("时间", ascending=False).head()

In [ ]:
slice_source("12378明细", start_time, end_time).sort_values("时间", ascending=False).head()

In [ ]:
slice_source("常规", start_time, end_time).sort_values("时间", ascending=False).head()

In [ ]:
slice_source("贷后", start_time, end_time).sort_values("时间", ascending=False).head()

In [ ]:
slice_source("工单明细", start_time, end_time).sort_values("时间", ascending=False).head()

In [ ]:
slice_source("会话记录", start_time, end_time).sort_values("时间", ascending=False).head()